<!-- CELL-00 -->
# PRD-300 / CC-1.5.1 — NLP Backfill pe GPU

Rulează pipeline-ul rhetoric (FOMC-RoBERTa + Llama 3.3 70B) pe toate documentele FOMC (statements + minutes).

**Cerințe runtime:**
- Google Colab cu GPU (Runtime → Change runtime type → T4 GPU)
- Secrets setate în Colab (icon 🔑 sidebar stânga):
  - `HF_TOKEN` — token HuggingFace (hf_...)
  - `DEEPINFRA_API_KEY` — cheie API DeepInfra

**Timp estimat:** ~20-30 min (RoBERTa pe GPU: ~18 min, Llama API: ~15 min per statements, minutes din cache)

**Output:** `data/rhetoric/fomc_scores.parquet` — descarcă la final și pune în repo local.

In [ ]:
# CELL-01
print("[CELL-01]")

print("[CELL-01] Environment setup + GPU check")

import torch

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    gpu_mem = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print(f"GPU available: {gpu_name} ({gpu_mem:.1f} GB)")
else:
    print("NO GPU -- Runtime -> Change runtime type -> T4 GPU")
    print("  RoBERTa will run on CPU (~6 hours instead of ~18 minutes)")

# Verify Colab secrets
try:
    from google.colab import userdata
    hf_token = userdata.get("HF_TOKEN")
    deepinfra_key = userdata.get("DEEPINFRA_API_KEY")

    if hf_token and hf_token.startswith("hf_"):
        print(f"HF_TOKEN: valid (hf_...{hf_token[-4:]})")
    else:
        print(f"HF_TOKEN: wrong prefix or missing -- needs hf_ prefix")

    if deepinfra_key and len(deepinfra_key) > 10:
        print(f"DEEPINFRA_API_KEY: present ({len(deepinfra_key)} chars)")
    else:
        print(f"DEEPINFRA_API_KEY: missing or short -- Llama scorer will be skipped")

    # Set as env vars for the pipeline
    import os
    os.environ["HF_TOKEN"] = hf_token or ""
    os.environ["DEEPINFRA_API_KEY"] = deepinfra_key if deepinfra_key else ""
    print("\nSecrets loaded into environment")

except Exception as e:
    print(f"Colab secrets error: {e}")
    print("  Go to sidebar key icon -> add HF_TOKEN and DEEPINFRA_API_KEY")

In [ ]:
# CELL-02
print("[CELL-02]")
print("[CELL-02] Clone repo + install")

import os
import sys
import subprocess

REPO_URL = "https://github.com/Inocenthhacker/macro_context_reader.git"
REPO_DIR = "/content/macro_context_reader"

if os.path.exists(REPO_DIR):
    print(f"Repo already cloned at {REPO_DIR}")
    os.chdir(REPO_DIR)
    !git pull --quiet
    print("Pulled latest changes")
else:
    !git clone {REPO_URL} {REPO_DIR} --quiet
    os.chdir(REPO_DIR)
    print(f"Cloned to {REPO_DIR}")

# Install project
subprocess.check_call([sys.executable, "-m", "pip", "install", "-e", ".", "--quiet"])

# Force Python to find src/ layout
if "/content/macro_context_reader/src" not in sys.path:
    sys.path.insert(0, "/content/macro_context_reader/src")

# Clear stale namespace package cache (critical - Colab caches macro_context_reader as
# empty namespace package on first import attempt before sys.path is set)
for mod in list(sys.modules.keys()):
    if "macro_context_reader" in mod:
        del sys.modules[mod]

# Now import cleanly
from macro_context_reader.rhetoric.pipeline import run_full_pipeline
from macro_context_reader.rhetoric.scraper import fetch_fomc_statements, fetch_fomc_minutes
print("Pipeline imports OK")

# Show cached documents
from pathlib import Path
statements = list(Path("data/rhetoric/raw/statement").glob("*")) if Path("data/rhetoric/raw/statement").exists() else []
minutes = list(Path("data/rhetoric/raw/minutes").glob("*")) if Path("data/rhetoric/raw/minutes").exists() else []
print(f"Cached documents: {len(statements)} statements, {len(minutes)} minutes")

<!-- CELL-03 -->
## Pas 1: Verificare acces FOMC-RoBERTa

Descarcă modelul pe GPU. Prima rulare descarcă ~1.4 GB, ulterior e din cache.

In [ ]:
# CELL-04
print("[CELL-04]")

print("[CELL-03] Load FOMC-RoBERTa on GPU")

import os
import torch
from transformers import AutoModelForSequenceClassification, AutoTokenizer

token = os.environ.get("HF_TOKEN")
device = 0 if torch.cuda.is_available() else -1

print(f"Loading FOMC-RoBERTa (device={'GPU' if device == 0 else 'CPU'})...")
tokenizer = AutoTokenizer.from_pretrained("gtfintechlab/FOMC-RoBERTa", token=token)
model = AutoModelForSequenceClassification.from_pretrained("gtfintechlab/FOMC-RoBERTa", token=token)

if device == 0:
    model = model.cuda()

# Quick sanity test
inputs = tokenizer("Inflation remains elevated", return_tensors="pt", truncation=True)
if device == 0:
    inputs = {k: v.cuda() for k, v in inputs.items()}
with torch.no_grad():
    outputs = model(**inputs)
probs = torch.softmax(outputs.logits, dim=-1).squeeze().cpu().tolist()
print(f"Model loaded. Test: 'Inflation remains elevated' -> hawkish: {probs[1]:.3f}")

del model, tokenizer, inputs, outputs  # free memory for pipeline
torch.cuda.empty_cache()

<!-- CELL-05 -->
## Pas 2: Scrape documente lipsă (dacă e cazul)

Verifică că avem toate documentele cached. Dacă lipsesc, scrape-uiește.

In [ ]:
# CELL-06
print("[CELL-06]")

print("[CELL-04] Ensure all docs cached")

from macro_context_reader.rhetoric.scraper import fetch_fomc_statements, fetch_fomc_minutes
from pathlib import Path

# Check current cache
stmt_dir = Path("data/rhetoric/raw/statement")
mins_dir = Path("data/rhetoric/raw/minutes")

stmt_count = len(list(stmt_dir.glob("*"))) if stmt_dir.exists() else 0
mins_count = len(list(mins_dir.glob("*"))) if mins_dir.exists() else 0

print(f"Current cache: {stmt_count} statements, {mins_count} minutes")

if stmt_count < 40:
    print("Fetching statements...")
    stmts = fetch_fomc_statements(start_year=2021)
    print(f"  Fetched {len(stmts)} statements")
else:
    print("Statements cache OK")

if mins_count < 40:
    print("Fetching minutes...")
    mins = fetch_fomc_minutes(start_year=2021)
    print(f"  Fetched {len(mins)} minutes")
else:
    print("Minutes cache OK")

# Final count
stmt_count = len(list(stmt_dir.glob("*"))) if stmt_dir.exists() else 0
mins_count = len(list(mins_dir.glob("*"))) if mins_dir.exists() else 0
print(f"\nTotal cached: {stmt_count} statements + {mins_count} minutes = {stmt_count + mins_count} documents")

<!-- CELL-07 -->
## Pas 3: RoBERTa scoring (rapid, GPU)

FOMC-RoBERTa pe GPU pentru toate 84 documente. Durează ~15-20 min.
Output intermediar: `data/rhetoric/fomc_scores_roberta.parquet`

In [ ]:
# CELL-08
print("[CELL-08]")
print("[CELL-08] RoBERTa-only pipeline on all documents (Drive-persistent)")

import time
import logging
from pathlib import Path

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s %(levelname)s %(name)s: %(message)s",
    datefmt="%H:%M:%S",
)

# Mount Drive (idempotent — if CELL-10 already mounted, no-op)
try:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    DRIVE_BASE = Path("/content/drive/MyDrive/macro_context_reader")
    DRIVE_BASE.mkdir(parents=True, exist_ok=True)
    print(f"Drive mounted: {DRIVE_BASE}")
except Exception as e:
    print(f"WARNING: Drive mount failed ({e}). Falling back to local /content (NOT persistent across sessions).")
    DRIVE_BASE = Path("/content/macro_context_reader_persistent")
    DRIVE_BASE.mkdir(parents=True, exist_ok=True)

# Persistent paths
drive_roberta = DRIVE_BASE / "fomc_scores_roberta.parquet"
local_roberta = Path("data/rhetoric/fomc_scores_roberta.parquet")

# SKIP-IF-EXISTS: if Drive parquet exists, just copy to local and skip the 22-min pipeline run
if drive_roberta.exists():
    import pandas as pd
    import shutil

    print(f"\n*** SKIPPED — RoBERTa parquet found in Drive ***")
    print(f"Source: {drive_roberta}")

    # Copy from Drive to local (so other cells see it at expected path)
    local_roberta.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy(drive_roberta, local_roberta)
    print(f"Copied to local: {local_roberta}")

    # Load and display summary (so user sees what was loaded)
    result_roberta = pd.read_parquet(local_roberta)
    print(f"\nLoaded {len(result_roberta)} previously-scored documents")
    print(f"Columns: {result_roberta.columns.tolist()}")
    if len(result_roberta) > 0:
        print(f"Date range: {result_roberta['date'].min()} to {result_roberta['date'].max()}")
        if "doc_type" in result_roberta.columns:
            print(f"\nBy doc_type:")
            print(result_roberta["doc_type"].value_counts().to_string())
        score_cols = [c for c in result_roberta.columns if "net" in c.lower()]
        for col in score_cols:
            if result_roberta[col].dtype in ["float64", "float32"]:
                print(f"\n{col}: min={result_roberta[col].min():.3f}, max={result_roberta[col].max():.3f}, mean={result_roberta[col].mean():.3f}")
    print(f"\nSkip rationale: re-running RoBERTa pipeline produces identical scores (deterministic).")
    print(f"To force re-run: delete {drive_roberta} and re-execute this cell.")
else:
    # No cached output — run the pipeline
    print(f"\n*** RUNNING — no RoBERTa parquet in Drive ***")
    print(f"Will save to: {drive_roberta}")
    print(f"Estimated time: ~20-25 minutes on T4 GPU\n")

    from macro_context_reader.rhetoric.pipeline import run_full_pipeline

    start = time.time()

    result_roberta = run_full_pipeline(
        start_year=2021,
        doc_types=["statement", "minutes"],
        scorer_names=["fomc_roberta"],
        force_refetch=False,
        output_path=local_roberta,
    )

    elapsed = time.time() - start

    print(f"\n{'='*60}")
    print(f"RoBERTa COMPLETE — {elapsed/60:.1f} minutes")
    print(f"{'='*60}")
    print(f"Documents scored: {len(result_roberta)}")
    print(f"Columns: {result_roberta.columns.tolist()}")
    print(f"Saved to local: {local_roberta}")

    # Persist to Drive
    import shutil
    shutil.copy(local_roberta, drive_roberta)
    print(f"Saved to Drive: {drive_roberta}")
    print(f"  (Future sessions will skip this cell automatically)")

    if len(result_roberta) > 0:
        print(f"\nDate range: {result_roberta['date'].min()} to {result_roberta['date'].max()}")
        if "doc_type" in result_roberta.columns:
            print(f"\nBy doc_type:")
            print(result_roberta["doc_type"].value_counts().to_string())
        score_cols = [c for c in result_roberta.columns if "net" in c.lower()]
        for col in score_cols:
            if result_roberta[col].dtype in ["float64", "float32"]:
                print(f"\n{col}: min={result_roberta[col].min():.3f}, max={result_roberta[col].max():.3f}, mean={result_roberta[col].mean():.3f}")

<!-- CELL-09 -->
## Pas 4: Llama scoring în 3 chunks

Llama 3.3 70B prin DeepInfra API. ~9 ore total împărțit în 3 chunks de 28 documente.

**Pauză de 30 secunde între chunks** — în acel interval poți opri pipeline-ul (Runtime → Interrupt) dacă e nevoie. Cache-ul Llama persistă între chunks, deci re-pornirea continuă de unde ai rămas.

**Chunks:**
- Chunk 1: documente 1-28 (~3 ore)
- Chunk 2: documente 29-56 (~3 ore)
- Chunk 3: documente 57-84 (~3 ore)

In [ ]:
# CELL-10
print("[CELL-10]")
print("[CELL-10] Llama scoring in 3 chunks with Drive persistence + auto-resume")

import os
import sys
import time
from pathlib import Path
from datetime import datetime, timedelta

# Mount Google Drive (idempotent — already mounted = no-op)
try:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    DRIVE_BASE = Path("/content/drive/MyDrive/macro_context_reader")
    DRIVE_BASE.mkdir(parents=True, exist_ok=True)
    print(f"Drive mounted: {DRIVE_BASE}")
except Exception as e:
    print(f"WARNING: Drive mount failed ({e}). Falling back to local /content storage (NOT persistent).")
    DRIVE_BASE = Path("/content/macro_context_reader_persistent")
    DRIVE_BASE.mkdir(parents=True, exist_ok=True)

# Persistent paths
DRIVE_LLAMA_CACHE = DRIVE_BASE / "llama_cache"
DRIVE_CHUNKS_DIR = DRIVE_BASE / "chunks"
DRIVE_LLAMA_CACHE.mkdir(parents=True, exist_ok=True)
DRIVE_CHUNKS_DIR.mkdir(parents=True, exist_ok=True)

# Symlink local llama_cache to Drive cache (so pipeline reads/writes to Drive transparently)
local_cache = Path("data/rhetoric/llama_cache")
local_cache.parent.mkdir(parents=True, exist_ok=True)
if local_cache.exists() and not local_cache.is_symlink():
    import shutil
    # Migrate existing local cache files to Drive
    for f in local_cache.glob("*"):
        target = DRIVE_LLAMA_CACHE / f.name
        if not target.exists():
            shutil.copy(f, target)
    shutil.rmtree(local_cache)
if not local_cache.exists():
    local_cache.symlink_to(DRIVE_LLAMA_CACHE)
print(f"Llama cache symlinked: {local_cache} -> {DRIVE_LLAMA_CACHE}")
print(f"  Existing cache entries: {len(list(DRIVE_LLAMA_CACHE.glob('*')))}")

# Load all documents
from macro_context_reader.rhetoric.scraper import fetch_fomc_statements, fetch_fomc_minutes
from macro_context_reader.rhetoric.pipeline import run_full_pipeline
import pandas as pd

print("\nLoading document list...")
all_statements = fetch_fomc_statements(start_year=2021)
all_minutes = fetch_fomc_minutes(start_year=2021)
all_docs = sorted(all_statements + all_minutes, key=lambda d: (d.date, d.doc_type, d.url))
total_docs = len(all_docs)
print(f"Total documents: {total_docs} ({len(all_statements)} statements + {len(all_minutes)} minutes)")

# Split into 3 chunks
N_CHUNKS = 3
chunk_size = (total_docs + N_CHUNKS - 1) // N_CHUNKS
chunks = [all_docs[i:i + chunk_size] for i in range(0, total_docs, chunk_size)]
print(f"\nChunk plan ({chunk_size} docs/chunk):")
for i, chunk in enumerate(chunks, 1):
    print(f"  Chunk {i}: {len(chunk)} docs ({chunk[0].date.date()} to {chunk[-1].date.date()})")

# Identify which docs are already processed (from existing chunk parquets in Drive)
processed_keys = set()
for chunk_file in DRIVE_CHUNKS_DIR.glob("fomc_scores_llama_chunk*.parquet"):
    try:
        df_existing = pd.read_parquet(chunk_file)
        for _, row in df_existing.iterrows():
            key = (str(row["date"]), row["doc_type"], row["doc_url"])
            processed_keys.add(key)
        print(f"Loaded {len(df_existing)} processed docs from {chunk_file.name}")
    except Exception as e:
        print(f"WARNING: Could not read {chunk_file}: {e}")

print(f"\n*** RESUME STATE ***")
print(f"Already processed: {len(processed_keys)} / {total_docs} docs ({len(processed_keys)/total_docs*100:.1f}%)")

# Live progress display function
def render_progress(chunk_idx, n_chunks, doc_in_chunk, chunk_total, doc_global, total_global, eta_str=""):
    chunk_pct = (doc_in_chunk / chunk_total * 100) if chunk_total else 0
    global_pct = (doc_global / total_global * 100) if total_global else 0
    bar_chunk = "#" * int(chunk_pct / 5) + "-" * (20 - int(chunk_pct / 5))
    bar_global = "#" * int(global_pct / 5) + "-" * (20 - int(global_pct / 5))
    print(f"\n{'='*70}")
    print(f"CHUNK {chunk_idx}/{n_chunks}  [{bar_chunk}] {chunk_pct:.0f}% ({doc_in_chunk}/{chunk_total})")
    print(f"GLOBAL       [{bar_global}] {global_pct:.0f}% ({doc_global}/{total_global})  {eta_str}")
    print(f"{'='*70}")

# Process chunks
CHECKPOINT_EVERY = 10  # save parquet after every N docs
total_start = time.time()
total_processed_this_session = 0

for chunk_idx, chunk_docs in enumerate(chunks, 1):
    chunk_start = time.time()
    chunk_dates = [d.date for d in chunk_docs]
    chunk_start_date = min(chunk_dates).date()
    chunk_end_date = max(chunk_dates).date()

    chunk_parquet = DRIVE_CHUNKS_DIR / f"fomc_scores_llama_chunk{chunk_idx}.parquet"

    # Filter chunk to only unprocessed docs
    chunk_to_process = [d for d in chunk_docs if (str(pd.Timestamp(d.date)), d.doc_type, d.url) not in processed_keys]

    print(f"\n{'#'*70}")
    print(f"# CHUNK {chunk_idx}/{N_CHUNKS}")
    print(f"# Range: {chunk_start_date} to {chunk_end_date}")
    print(f"# Total in chunk: {len(chunk_docs)}, already done: {len(chunk_docs) - len(chunk_to_process)}, TO PROCESS: {len(chunk_to_process)}")
    print(f"{'#'*70}")

    if len(chunk_to_process) == 0:
        print(f"Chunk {chunk_idx} fully processed already — skipping")
    else:
        # Process docs one by one with progress display
        chunk_results = []
        # Load existing chunk parquet if it exists (to append to)
        if chunk_parquet.exists():
            try:
                existing_df = pd.read_parquet(chunk_parquet)
                chunk_results = existing_df.to_dict("records")
                print(f"Loaded {len(chunk_results)} existing rows from chunk parquet")
            except Exception as e:
                print(f"WARNING: Could not load existing chunk parquet: {e}")

        for doc_idx, doc in enumerate(chunk_to_process, 1):
            doc_start = time.time()

            # Calculate global position for progress display
            doc_global = len(processed_keys) + total_processed_this_session + 1

            # ETA calculation
            if total_processed_this_session > 0:
                elapsed_session = time.time() - total_start
                rate = total_processed_this_session / elapsed_session  # docs/sec
                remaining_global = total_docs - doc_global
                eta_sec = remaining_global / rate if rate > 0 else 0
                eta_str = f"ETA: {timedelta(seconds=int(eta_sec))}"
            else:
                eta_str = "ETA: calculating..."

            render_progress(chunk_idx, N_CHUNKS, doc_idx, len(chunk_to_process),
                            doc_global, total_docs, eta_str)
            print(f"Processing: {doc.date.date()} | {doc.doc_type} | {doc.url[:80]}...")

            try:
                # Use pipeline with single-day window targeting this specific doc
                doc_date = doc.date.date() if hasattr(doc.date, "date") else doc.date
                result_doc = run_full_pipeline(
                    start_year=doc_date.year,
                    end_date=doc_date,
                    doc_types=[doc.doc_type],
                    scorer_names=["llama_deepinfra"],
                    force_refetch=False,
                    output_path=Path("/tmp/llama_per_doc_scratch.parquet"),  # required by pipeline; we ignore output and use returned DataFrame
                )

                if len(result_doc) > 0:
                    # Filter to just this doc (start_year=year may include other docs from same year)
                    result_doc = result_doc[result_doc["doc_url"] == doc.url]
                    if len(result_doc) > 0:
                        chunk_results.extend(result_doc.to_dict("records"))
                        total_processed_this_session += 1
                        processed_keys.add((str(pd.Timestamp(doc.date)), doc.doc_type, doc.url))
                        elapsed = time.time() - doc_start
                        print(f"  Done in {elapsed:.1f}s  ({len(result_doc)} row added)")
                    else:
                        print(f"  WARNING: doc {doc.url} not in pipeline output")
                else:
                    print(f"  WARNING: empty result for {doc.url}")

                # Checkpoint every CHECKPOINT_EVERY docs
                if total_processed_this_session > 0 and total_processed_this_session % CHECKPOINT_EVERY == 0:
                    df_checkpoint = pd.DataFrame(chunk_results)
                    df_checkpoint = df_checkpoint.drop_duplicates(subset=["date", "doc_type", "doc_url"], keep="last")
                    df_checkpoint.to_parquet(chunk_parquet, engine="pyarrow", compression="snappy", index=False)
                    print(f"  >>> CHECKPOINT: saved {len(df_checkpoint)} rows to {chunk_parquet.name}")

            except Exception as e:
                print(f"  ERROR on doc {doc.url}: {type(e).__name__}: {e}")
                import traceback
                traceback.print_exc()
                # Continue with next doc — cache preserves what was done

        # Final save for this chunk
        if chunk_results:
            df_chunk_final = pd.DataFrame(chunk_results)
            df_chunk_final = df_chunk_final.drop_duplicates(subset=["date", "doc_type", "doc_url"], keep="last")
            df_chunk_final.to_parquet(chunk_parquet, engine="pyarrow", compression="snappy", index=False)
            print(f"\n>>> CHUNK {chunk_idx} FINAL SAVE: {len(df_chunk_final)} rows to {chunk_parquet.name}")

        chunk_elapsed = time.time() - chunk_start
        print(f"\nChunk {chunk_idx} complete: {chunk_elapsed/60:.1f} min")

    # Pause between chunks (except after last)
    if chunk_idx < N_CHUNKS:
        print(f"\n{'='*70}")
        print(f"PAUZA 30 SECUNDE — Runtime > Interrupt acum daca vrei sa opresti")
        print(f"{'='*70}")
        for remaining in range(30, 0, -5):
            print(f"  Continua in {remaining} secunde...")
            time.sleep(5)

total_elapsed = time.time() - total_start
print(f"\n{'='*70}")
print(f"ALL CHUNKS COMPLETE — session time: {total_elapsed/60:.1f} min")
print(f"Documents processed this session: {total_processed_this_session}")
print(f"{'='*70}")

# Combine all chunk parquets into single Llama output (in Drive AND local)
print("\nCombining chunks into final fomc_scores_llama.parquet...")
chunk_files = sorted(DRIVE_CHUNKS_DIR.glob("fomc_scores_llama_chunk*.parquet"))
print(f"Found {len(chunk_files)} chunk parquets in Drive")

if chunk_files:
    dfs = [pd.read_parquet(f) for f in chunk_files]
    combined_llama = pd.concat(dfs, ignore_index=True)
    combined_llama = combined_llama.drop_duplicates(subset=["date", "doc_type", "doc_url"], keep="last")
    combined_llama = combined_llama.sort_values("date").reset_index(drop=True)

    # Save to Drive (persistent) + local (for download)
    drive_combined = DRIVE_BASE / "fomc_scores_llama.parquet"
    local_combined = Path("data/rhetoric/fomc_scores_llama.parquet")

    combined_llama.to_parquet(drive_combined, engine="pyarrow", compression="snappy", index=False)
    combined_llama.to_parquet(local_combined, engine="pyarrow", compression="snappy", index=False)

    print(f"Combined: {len(combined_llama)} docs")
    print(f"  Drive:  {drive_combined}")
    print(f"  Local:  {local_combined}")
else:
    print("No chunk parquets found — nothing to combine")

<!-- CELL-11 -->
## Pas 5: Combinare RoBERTa + Llama

Unește scorurile RoBERTa și Llama într-un singur parquet `fomc_scores.parquet`.

In [ ]:
# CELL-12
print("[CELL-12]")
print("[CELL-12] Combine RoBERTa + Llama into final parquet")

import pandas as pd
from pathlib import Path

roberta_path = Path("data/rhetoric/fomc_scores_roberta.parquet")
llama_path = Path("data/rhetoric/fomc_scores_llama.parquet")
final_path = Path("data/rhetoric/fomc_scores.parquet")

if not roberta_path.exists():
    raise FileNotFoundError(f"RoBERTa parquet missing: {roberta_path}")
if not llama_path.exists():
    raise FileNotFoundError(f"Llama parquet missing: {llama_path}")

df_roberta = pd.read_parquet(roberta_path)
df_llama = pd.read_parquet(llama_path)

print(f"RoBERTa: {len(df_roberta)} docs, columns: {df_roberta.columns.tolist()}")
print(f"Llama:   {len(df_llama)} docs, columns: {df_llama.columns.tolist()}")

# Identify columns unique to each scorer
roberta_score_cols = [c for c in df_roberta.columns if 'roberta' in c.lower() or 'fomc_roberta' in c.lower()]
llama_score_cols = [c for c in df_llama.columns if 'llama' in c.lower() or 'deepinfra' in c.lower()]

# Keep common keys + score columns from each
join_keys = ['date', 'doc_type', 'doc_url']
common_cols = [c for c in df_roberta.columns if c in df_llama.columns and c not in join_keys]

# Merge on join keys, keeping scorer-specific columns from each side
df_combined = df_roberta.merge(
    df_llama[join_keys + llama_score_cols],
    on=join_keys,
    how='outer',
    suffixes=('_roberta_meta', '_llama_meta'),
)

print(f"\nCombined: {len(df_combined)} docs, columns: {df_combined.columns.tolist()}")

df_combined.to_parquet(final_path, engine='pyarrow', compression='snappy', index=False)
print(f"\nSaved: {final_path}")
print(f"Size: {final_path.stat().st_size / 1024:.1f} KB")

<!-- CELL-13 -->
## Pas 6: Validare + Spot checks

In [ ]:
# CELL-14
print("[CELL-14]")

print("[CELL-06] Validation + spot checks")

import pandas as pd

df = pd.read_parquet("data/rhetoric/fomc_scores.parquet")

print(f"Shape: {df.shape}")
print(f"Columns: {df.columns.tolist()}")
print(f"Date range: {df['date'].min()} to {df['date'].max()}")
null_counts = df.isna().sum()
if null_counts.any():
    print(f"Null counts:\n{null_counts[null_counts > 0]}")
else:
    print("No nulls")

# Spot checks
checks = {
    "2022-03-16": "hawkish (hiking cycle start)",
    "2024-09-18": "dovish (cutting cycle start)",
}

for date_str, expected in checks.items():
    ts = pd.Timestamp(date_str)
    match = df[df['date'] == ts]
    if len(match) > 0:
        row = match.iloc[0]
        print(f"\n{'='*50}")
        print(f"Spot check: {date_str} -- expected {expected}")
        for col in ['ensemble_net', 'weighted_score', 'confidence']:
            if col in row.index:
                print(f"  {col}: {row[col]}")
        net_cols = [c for c in row.index if '_net' in c]
        for col in net_cols:
            print(f"  {col}: {row[col]:.3f}")
    else:
        print(f"\n{date_str} not found in data")

# Coverage check
if 'doc_type' in df.columns:
    stmt_count = (df['doc_type'] == 'statement').sum()
    mins_count = (df['doc_type'] == 'minutes').sum()
    print(f"\nCoverage: {stmt_count} statements + {mins_count} minutes = {len(df)} total")
    if len(df) >= 60:
        print(f"Meets >=60 threshold")
    else:
        print(f"Below >=60 threshold (got {len(df)})")

<!-- CELL-15 -->
## Pas 7: Descarcă parquet

Descarcă `fomc_scores.parquet` și pune-l în `data/rhetoric/` din repo-ul local.
Apoi rulează local:
```bash
git add -f data/rhetoric/fomc_scores.parquet
git commit -m "feat(rhetoric): NLP backfill -- 84 docs scored on Colab T4 GPU [PRD-300/CC-1.5.1]"
git push origin main
```

In [ ]:
# CELL-16
print("[CELL-16]")

print("[CELL-07] Download parquet")

from pathlib import Path

parquet_path = Path("data/rhetoric/fomc_scores.parquet")

if parquet_path.exists():
    size_kb = parquet_path.stat().st_size / 1024
    print(f"File: {parquet_path}")
    print(f"Size: {size_kb:.1f} KB")

    try:
        from google.colab import files
        files.download(str(parquet_path))
        print("Download triggered -- check your browser downloads")
    except ImportError:
        print("Not running in Colab -- file available at:")
        print(f"  {parquet_path.resolve()}")
else:
    print("Parquet not found -- pipeline may have failed")